# Write ATST example 3

Build a fully linked four-readout ATST package for four shaking speeds.

## Setup

Import the ATST block classes, multi-readout writer, Tecan reader, and data utilities.

In [1]:
from pathlib import Path

import pandas as pd

from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    ReadoutIds,
    Study,
)
from ATST import TecanReader


## Input data

Read each Tecan export and its corresponding layout.

In [2]:
reader = TecanReader()
settings = [("OD600_1", 100), ("OD600_2", 150), ("OD600_3", 300), ("OD600_4", 450)]
device_outputs = {
    readout_id: reader.read(
        Path(f"raw_outputs/simulated_tecan_{rpm}rpm_output.xlsx").read_bytes(),
        filename=f"simulated_tecan_{rpm}rpm_output.xlsx",
    )
    for readout_id, rpm in settings
}
layout_data = {
    readout_id: pd.read_csv(
        f"layout_inputs/layout_{readout_id}.tsv", sep="\t", keep_default_na=False
    )
    for readout_id, _ in settings
}

device_outputs["OD600_1"].readings.head()


,Time,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12
0,0,0.179,0.139,0.163,0.164,0.16,0.146,0.17,0.169,0.161,...,0.157,0.149,0.161,0.176,0.156,0.156,0.158,0.109,0.093,0.09
1,2400,0.163,0.187,0.172,0.179,0.164,0.149,0.168,0.154,0.15,...,0.146,0.141,0.159,0.17,0.17,0.168,0.162,0.094,0.088,0.115
2,4800,0.213,0.203,0.187,0.181,0.192,0.189,0.201,0.223,0.191,...,0.157,0.171,0.134,0.16,0.187,0.192,0.186,0.112,0.113,0.09
3,7200,0.232,0.219,0.226,0.207,0.233,0.224,0.25,0.223,0.22,...,0.157,0.158,0.163,0.161,0.235,0.218,0.219,0.095,0.111,0.111
4,9600,0.301,0.27,0.288,0.27,0.276,0.267,0.26,0.271,0.274,...,0.157,0.152,0.147,0.166,0.263,0.285,0.273,0.107,0.099,0.096


## FILE_INFO

`FILE_INFO` identifies the output file and ATST format. The file name, format, version, creation date, delimiter, and encoding are required.

In [3]:
file_info = FileMetadata(data={
    "file_name": "example_3.atst.txt",
    "format": "ATST",
    "format_version": "0.1",
    "created_on": "2026-01-01",
    "field_delimiter": "TAB",
    "encoding": "UTF-8",
})


## STUDY

`STUDY` identifies and describes the experiment. A non-empty title and study ID are required.

In [4]:
study = Study(data={
    "title": "Example 3 simulated shaking-speed series",
    "study_id": "ATST_EXAMPLE_3",
    "description": "Four fully linked OD600 readouts at 37C and different shaking speeds.",
})


## READOUT_IDS

`READOUT_IDS` lists the unique IDs used to connect each readout-specific block. It is required for a multi-readout ATST file.

In [5]:
readout_ids = ReadoutIds(data=pd.DataFrame({"readout_id": ["OD600_1", "OD600_2", "OD600_3", "OD600_4"]}))


## METADATA: OD600_1

This `METADATA` block records acquisition details for the 100 rpm readout.

In [6]:
od600_1_metadata = Metadata(data=device_outputs["OD600_1"].metadata)


## ASSAY: OD600_1

This `ASSAY` block defines the OD600 signal acquired at 100 rpm.

In [7]:
od600_1_assay = Assay(data=device_outputs["OD600_1"].assay)


## METADATA: OD600_2

This `METADATA` block records acquisition details for the 150 rpm readout.

In [8]:
od600_2_metadata = Metadata(data=device_outputs["OD600_2"].metadata)


## ASSAY: OD600_2

This `ASSAY` block defines the OD600 signal acquired at 150 rpm.

In [9]:
od600_2_assay = Assay(data=device_outputs["OD600_2"].assay)


## METADATA: OD600_3

This `METADATA` block records acquisition details for the 300 rpm readout.

In [10]:
od600_3_metadata = Metadata(data=device_outputs["OD600_3"].metadata)


## ASSAY: OD600_3

This `ASSAY` block defines the OD600 signal acquired at 300 rpm.

In [11]:
od600_3_assay = Assay(data=device_outputs["OD600_3"].assay)


## METADATA: OD600_4

This `METADATA` block records acquisition details for the 450 rpm readout.

In [12]:
od600_4_metadata = Metadata(data=device_outputs["OD600_4"].metadata)


## ASSAY: OD600_4

This `ASSAY` block defines the OD600 signal acquired at 450 rpm.

In [13]:
od600_4_assay = Assay(data=device_outputs["OD600_4"].assay)


## ENTITIES

`ENTITIES` defines the biological records referenced by the layout. Each table declares a primary key with unique, non-empty values. These tables should be generated before constructing the final ATST file.

In [14]:
phages = EntityTable(
    name="ENTITIES",
    table_name="PHAGES",
    pk="PHAGE_ID",
    data=pd.read_csv("entities/PHAGES.csv", keep_default_na=False),
)
isolates = EntityTable(
    name="ENTITIES",
    table_name="ISOLATES",
    pk="ISOLATE_ID",
    data=pd.read_csv("entities/ISOLATES.csv", keep_default_na=False),
)
entities = Entities(tables={"PHAGES": phages, "ISOLATES": isolates})


## LAYOUT: OD600_1

The 100 rpm layout maps its wells to the shared entity records.

In [15]:
od600_1_layout = Layout(data=layout_data["OD600_1"])


## READINGS: OD600_1

These are the OD600 time series measured at 100 rpm.

In [16]:
od600_1_readings = Readings(data=device_outputs["OD600_1"].readings)


## LAYOUT: OD600_2

The 150 rpm layout maps its wells to the shared entity records.

In [17]:
od600_2_layout = Layout(data=layout_data["OD600_2"])


## READINGS: OD600_2

These are the OD600 time series measured at 150 rpm.

In [18]:
od600_2_readings = Readings(data=device_outputs["OD600_2"].readings)


## LAYOUT: OD600_3

The 300 rpm layout maps its wells to the shared entity records.

In [19]:
od600_3_layout = Layout(data=layout_data["OD600_3"])


## READINGS: OD600_3

These are the OD600 time series measured at 300 rpm.

In [20]:
od600_3_readings = Readings(data=device_outputs["OD600_3"].readings)


## LAYOUT: OD600_4

The 450 rpm layout maps its wells to the shared entity records.

In [21]:
od600_4_layout = Layout(data=layout_data["OD600_4"])


## READINGS: OD600_4

These are the OD600 time series measured at 450 rpm.

In [22]:
od600_4_readings = Readings(data=device_outputs["OD600_4"].readings)


## Assemble and validate

Create one ATST object per readout, combine them, and validate the complete package.

In [23]:
od600_1 = ATSTFile(
    file_info=file_info, study=study, readout_ids=readout_ids,
    metadata=od600_1_metadata, assay=od600_1_assay, entities=entities,
    layout=od600_1_layout, readings=od600_1_readings,
)

od600_2 = ATSTFile(
    file_info=file_info, study=study, readout_ids=readout_ids,
    metadata=od600_2_metadata, assay=od600_2_assay, entities=entities,
    layout=od600_2_layout, readings=od600_2_readings,
)

od600_3 = ATSTFile(
    file_info=file_info, study=study, readout_ids=readout_ids,
    metadata=od600_3_metadata, assay=od600_3_assay, entities=entities,
    layout=od600_3_layout, readings=od600_3_readings,
)

od600_4 = ATSTFile(
    file_info=file_info, study=study, readout_ids=readout_ids,
    metadata=od600_4_metadata, assay=od600_4_assay, entities=entities,
    layout=od600_4_layout, readings=od600_4_readings,
)

atst = MultiReadoutATST(
    file_info, study, readout_ids,
    {"OD600_1": od600_1, "OD600_2": od600_2, "OD600_3": od600_3, "OD600_4": od600_4},
    entities,
)
validate_atst(atst)


## Write the ATST package

Write every payload block as a linked TSV file beside the container.

In [24]:
write_multi_readout_atst(atst, ".", linked_files=True)


PosixPath('example_3.atst.txt')